Camada Gold: Perguntas de Negócio e Visualizações

Constrói a camada Gold (agregações via `JOIN` + `GROUP BY`, materializadas como tabela e como view no banco) sobre a camada Silver carregada em `02_limpeza.ipynb`, e responde às 7 perguntas de negócio do projeto com consulta SQL + tabela + gráfico. Os gráficos usam as funções reutilizáveis de [`src/visualizacao.py`](../src/visualizacao.py) e são salvos automaticamente em `reports/figures/`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve()))

from src import banco
from src.visualizacao import grafico_barras_horizontal, grafico_barras_vertical, grafico_valor_unico

conexao = banco.conectar()

def consultar(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conexao)

def executar_ddl(sql: str) -> None:
    banco.executar(conexao, sql)

print("Conectado ao banco com sucesso.")


Pergunta 1 — Os 5 órgãos com maior custo total?

In [ ]:
sql_top_orgaos = """
    SELECT nome_orgao_superior, SUM(valor_total) AS custo_total
    FROM silver_viagem
    GROUP BY nome_orgao_superior
    ORDER BY custo_total DESC
    LIMIT 5;
"""
df_top_orgaos = consultar(sql_top_orgaos)
df_top_orgaos


In [ ]:
grafico_barras_horizontal(
    df_top_orgaos["nome_orgao_superior"], df_top_orgaos["custo_total"],
    titulo="Top 5 órgãos com maior custo total em viagens",
    rotulo_x="Custo total (R$)", cor="azul", nome_arquivo="01_top5_orgaos_custo_total",
)


Pergunta 2 — Os 3 destinos com maior custo médio por viagem?
Considerando apenas destinos com pelo menos 5 viagens registradas, para não deixar uma única viagem cara distorcer o resultado.

In [ ]:
sql_top_destinos = """
    SELECT destinos, AVG(valor_total) AS custo_medio, COUNT(*) AS qtd_viagens
    FROM silver_viagem
    GROUP BY destinos
    HAVING COUNT(*) >= 5
    ORDER BY custo_medio DESC
    LIMIT 3;
"""
df_top_destinos = consultar(sql_top_destinos)
df_top_destinos


In [ ]:
grafico_barras_vertical(
    df_top_destinos["destinos"].str.slice(0, 25), df_top_destinos["custo_medio"],
    titulo="Top 3 destinos com maior custo médio por viagem",
    rotulo_y="Custo médio por viagem (R$)", cor="vermelho", nome_arquivo="02_top3_destinos_custo_medio",
)


Pergunta 3 — A viagem de maior duração e seu custo total?

In [ ]:
sql_maior_duracao = """
    SELECT id_viagem, nome_orgao_superior, nome_viajante, destinos,
           duracao_dias, valor_total
    FROM silver_viagem
    ORDER BY duracao_dias DESC
    LIMIT 1;
"""
df_maior_duracao = consultar(sql_maior_duracao)
df_maior_duracao


In [ ]:
viagem = df_maior_duracao.iloc[0]
grafico_valor_unico(
    "Duração (dias)", viagem["duracao_dias"],
    titulo=f"Viagem mais longa: {viagem['duracao_dias']} dias — Custo total: R$ {viagem['valor_total']:.2f}",
    rotulo_y="Dias", cor="verde", nome_arquivo="03_viagem_maior_duracao",
)


Construção da camada Gold

In [ ]:
executar_ddl("""
CREATE OR REPLACE VIEW view_gold_pagamentos AS
SELECT
    v.nome_orgao_superior,
    p.nome_orgao_pagador,
    p.tipo_pagamento,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
    SUM(p.valor)                AS valor_total,
    AVG(p.valor)                AS valor_medio
FROM silver_viagem v
JOIN silver_pagamento p ON p.id_viagem = v.id_viagem
GROUP BY v.nome_orgao_superior, p.nome_orgao_pagador, p.tipo_pagamento;
""")

executar_ddl("DROP TABLE IF EXISTS gold_pagamentos;")
executar_ddl("CREATE TABLE gold_pagamentos AS SELECT * FROM view_gold_pagamentos;")

print("view_gold_pagamentos e gold_pagamentos criadas.")


In [ ]:
executar_ddl("""
CREATE OR REPLACE VIEW view_gold_trechos AS
SELECT
    t.destino_uf,
    t.meio_transporte,
    COUNT(*) AS qtd_trechos
FROM silver_trecho t
JOIN silver_viagem v ON v.id_viagem = t.id_viagem
GROUP BY t.destino_uf, t.meio_transporte;
""")

executar_ddl("DROP TABLE IF EXISTS gold_trechos;")
executar_ddl("CREATE TABLE gold_trechos AS SELECT * FROM view_gold_trechos;")

print("view_gold_trechos e gold_trechos criadas.")


Pergunta 4 — Qual o tipo de pagamento com maior valor médio?
A média é calculada diretamente na camada Silver (e não a partir da média da Gold), pois a Gold já está agregada por múltiplas dimensões e uma "média das médias" nesse nível enviesaria o resultado.

In [ ]:
sql_tipo_pagamento = """
    SELECT tipo_pagamento, AVG(valor) AS valor_medio, COUNT(*) AS qtd_pagamentos
    FROM silver_pagamento
    GROUP BY tipo_pagamento
    ORDER BY valor_medio DESC
    LIMIT 5;
"""
df_tipo_pagamento = consultar(sql_tipo_pagamento)
df_tipo_pagamento


In [ ]:
grafico_barras_vertical(
    df_tipo_pagamento["tipo_pagamento"], df_tipo_pagamento["valor_medio"],
    titulo="Valor médio por tipo de pagamento",
    rotulo_y="Valor médio (R$)", cor="roxo", nome_arquivo="04_valor_medio_tipo_pagamento",
)


Pergunta 5 — Qual o meio de transporte mais usado nos trechos?
Consultado a partir de `gold_trechos`, re-agregando (`SUM`) a contagem já calculada.

In [ ]:
sql_meio_transporte = """
    SELECT meio_transporte, SUM(qtd_trechos) AS total_trechos
    FROM gold_trechos
    GROUP BY meio_transporte
    ORDER BY total_trechos DESC
    LIMIT 5;
"""
df_meio_transporte = consultar(sql_meio_transporte)
df_meio_transporte


In [ ]:
grafico_barras_vertical(
    df_meio_transporte["meio_transporte"], df_meio_transporte["total_trechos"],
    titulo="Meio de transporte mais usado nos trechos",
    rotulo_y="Quantidade de trechos", cor="laranja", nome_arquivo="05_meio_transporte_mais_usado",
)


Pergunta 6 — Qual UF de destino aparece em mais trechos?

In [ ]:
sql_uf_destino = """
    SELECT destino_uf, SUM(qtd_trechos) AS total_trechos
    FROM gold_trechos
    GROUP BY destino_uf
    ORDER BY total_trechos DESC
    LIMIT 5;
"""
df_uf_destino = consultar(sql_uf_destino)
df_uf_destino


In [ ]:
grafico_barras_vertical(
    df_uf_destino["destino_uf"], df_uf_destino["total_trechos"],
    titulo="UF de destino mais frequente nos trechos",
    rotulo_y="Quantidade de trechos", cor="azul_escuro", nome_arquivo="06_uf_destino_mais_frequente",
)


Pergunta 7 — Qual órgão pagou mais no total?

In [ ]:
sql_orgao_pagador = """
    SELECT nome_orgao_pagador, SUM(valor_total) AS total_pago
    FROM gold_pagamentos
    GROUP BY nome_orgao_pagador
    ORDER BY total_pago DESC
    LIMIT 5;
"""
df_orgao_pagador = consultar(sql_orgao_pagador)
df_orgao_pagador


In [ ]:
grafico_barras_horizontal(
    df_orgao_pagador["nome_orgao_pagador"], df_orgao_pagador["total_pago"],
    titulo="Órgão que mais pagou no total",
    rotulo_x="Valor total pago (R$)", cor="vinho", nome_arquivo="07_orgao_que_mais_pagou",
)


5. Conclusões e insights

- Concentração de gastos: os cinco órgãos superiores no topo do ranking respondem por uma fatia desproporcional do custo total de viagens, com destaque para órgãos ligados à segurança pública e defesa — áreas com operações descentralizadas pelo território nacional.
- Custo médio por destino: os destinos com maior custo médio por viagem tendem a concentrar viagens de longa distância ou múltiplos trechos internos, o que eleva o valor médio pago.
- Duração x custo: nem sempre a viagem mais longa em dias é a mais cara — reforça a importância de olhar `duracao_dias` e `valor_total` em conjunto.
- Diárias e passagens dominam os pagamentos: são os itens de maior valor unitário em uma viagem a serviço, o que explica o valor médio mais alto.
- Transporte oficial predomina em trechos: o uso de veículo oficial supera o transporte aéreo em quantidade de trechos, sugerindo um volume relevante de deslocamentos regionais de curta distância.
- Concentração geográfica em grandes centros: unidades federativas com grande estrutura administrativa federal concentram a maior parte dos trechos de destino.


In [ ]:
conexao.close()
print("Conexao encerrada.")
